# Heavy Equipment Selling Price Prediction

This notebook predicts `TargetValue` using RMSLE as the primary metric. It is designed to run unchanged from the repository or in the Kaggle competition environment. Models are trained on `log1p(TargetValue)`, which aligns training with RMSLE and ensures non-negative predictions after inverse transformation.

In [ ]:
from pathlib import Path
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
MODE = 'debug'  # Use 'competition' only for the full Kaggle training workflow.
assert MODE in {'debug', 'competition'}
DEBUG_SAMPLE_SIZE = 30_000
TUNING_SAMPLE_SIZE = 50_000
TARGET = 'TargetValue'
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', palette='deep')

In [ ]:
# Resolve paths for both Kaggle and local execution.
KAGGLE_DATA = Path('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge')
LOCAL_DATA_CANDIDATES = [Path('../data'), Path('data')]
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.exists() else next((path for path in LOCAL_DATA_CANDIDATES if path.exists()), Path('../data'))
OUTPUT_DIR = Path('/kaggle/working') if KAGGLE_DATA.exists() else Path('../submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
META_PATH = DATA_DIR / 'metadata.csv'

assert TRAIN_PATH.exists() and TEST_PATH.exists(), f'Dataset not found in {DATA_DIR.resolve()}'
train_raw = pd.read_csv(TRAIN_PATH, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, low_memory=False)
metadata = pd.read_csv(META_PATH) if META_PATH.exists() else pd.DataFrame()
if MODE == 'debug':
    train_raw = train_raw.sample(n=min(DEBUG_SAMPLE_SIZE, len(train_raw)), random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Debug mode: using a reproducible {len(train_raw):,}-row training sample.')
else:
    print('Competition mode: using the complete training dataset.')

print(f'Train shape: {train_raw.shape}')
print(f'Test shape:  {test_raw.shape}')
display(train_raw.head())

## Data overview and missingness

The data combines dates, technical specifications, operational measures, and high-cardinality identifiers. Missingness is retained as signal: for used equipment, an unavailable meter or specification often conveys useful information rather than being random.

In [ ]:
overview = pd.DataFrame({
    'dtype': train_raw.dtypes.astype(str),
    'missing_count': train_raw.isna().sum(),
    'missing_pct': (100 * train_raw.isna().mean()).round(2),
    'n_unique': train_raw.nunique(dropna=True)
}).sort_values('missing_pct', ascending=False)
display(overview)

if not metadata.empty:
    display(metadata.head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.histplot(np.log1p(train_raw[TARGET]), bins=50, ax=axes[0], color='teal')
axes[0].set(title='Log target distribution', xlabel='log1p(TargetValue)')

missing_plot = overview.query('missing_pct > 0').head(20).sort_values('missing_pct')
axes[1].barh(missing_plot.index, missing_plot['missing_pct'], color='coral')
axes[1].set(title='Top 20 missing-value rates', xlabel='Missing (%)')

date_for_eda = pd.to_datetime(train_raw['TransactionDate'], errors='coerce')
yearly_target = train_raw.assign(TransactionYear=date_for_eda.dt.year).groupby('TransactionYear')[TARGET].median()
yearly_target.plot(marker='o', ax=axes[2], color='slateblue')
axes[2].set(title='Median price by transaction year', ylabel='Median TargetValue')
plt.tight_layout()
plt.show()

# These views show why a log target is appropriate, which fields need robust handling, and why transaction time is useful.

## Outlier analysis

Equipment prices and operational hours are naturally right-skewed. Rather than deleting valid high-value machines, the analysis flags extreme observations and the model uses log transformations and robust tree-based methods.

In [ ]:
outlier_columns = [column for column in [TARGET, 'ManufactureYear', 'OperationalHoursMeter'] if column in train_raw]
outlier_rows = []
for column in outlier_columns:
    values = pd.to_numeric(train_raw[column], errors='coerce').dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({'feature': column, 'q1': q1, 'q3': q3, 'lower_fence': lower,
                         'upper_fence': upper, 'iqr_outlier_count': ((values < lower) | (values > upper)).sum()})
outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

fig, axes = plt.subplots(1, len(outlier_columns), figsize=(6 * len(outlier_columns), 4))
axes = np.atleast_1d(axes)
for axis, column in zip(axes, outlier_columns):
    values = pd.to_numeric(train_raw[column], errors='coerce').clip(lower=0)
    sns.boxplot(x=np.log1p(values), ax=axis, color='skyblue')
    axis.set(title=f'log1p({column})', xlabel='')
plt.tight_layout()
plt.show()
# No rows are removed: IQR flags guide interpretation, while clipping and log features limit extreme-value influence.

## Feature engineering

All transformations are fitted only on the training partition. The engineered calendar, age, descriptor, availability, frequency, rarity, and interaction features provide compact signal without one-hot encoding every high-cardinality identifier.

In [ ]:
REQUIRED_FEATURES = ['TransactionYear', 'TransactionQuarter', 'AssetAge', 'DescriptorLength',
                     'HasOperationalHours', 'HasVariantModifier']

def add_base_features(frame):
    """Create deterministic features available in both train and test data."""
    df = frame.copy()
    transaction_date = pd.to_datetime(df['TransactionDate'], errors='coerce')
    df['TransactionYear'] = transaction_date.dt.year
    df['TransactionQuarter'] = transaction_date.dt.quarter
    df['TransactionMonth'] = transaction_date.dt.month
    df['TransactionWeek'] = transaction_date.dt.isocalendar().week.astype('float64')
    df['TransactionDay'] = transaction_date.dt.day
    df['TransactionDayOfWeek'] = transaction_date.dt.dayofweek
    df['TransactionSeason'] = ((df['TransactionMonth'] % 12) // 3 + 1).astype('float64')

    manufacture_year = pd.to_numeric(df.get('ManufactureYear'), errors='coerce')
    df['AssetAge'] = (df['TransactionYear'] - manufacture_year).clip(lower=0, upper=100)
    df['AssetAgeSquared'] = df['AssetAge'] ** 2

    descriptor = df.get('Spec_FullDescriptor', pd.Series('', index=df.index)).fillna('').astype(str)
    df['DescriptorLength'] = descriptor.str.len()
    df['DescriptorCharacterCount'] = df['DescriptorLength']
    df['DescriptorWordCount'] = descriptor.str.split().str.len()

    hours = pd.to_numeric(df.get('OperationalHoursMeter'), errors='coerce')
    df['HasOperationalHours'] = hours.notna().astype('int8')
    df['LogOperationalHours'] = np.log1p(hours.clip(lower=0))
    df['OperationalHoursPerYear'] = hours / df['AssetAge'].replace(0, np.nan)
    df['HasVariantModifier'] = df.get('Spec_VariantModifier', pd.Series(index=df.index)).notna().astype('int8')
    df['MissingFeatureCount'] = df.isna().sum(axis=1)

    for left, right, name in [('RegionCode', 'Spec_BaseClass', 'RegionBaseClass'),
                              ('VendorPartnerID', 'UtilizationTier', 'VendorUtilization')]:
        if left in df and right in df:
            df[name] = df[left].fillna('__MISSING__').astype(str) + '__' + df[right].fillna('__MISSING__').astype(str)
    return df.drop(columns=['TransactionDate'], errors='ignore')

def fit_category_maps(frame, categorical_columns, rare_min_count=20):
    frequency_maps, rare_values = {}, {}
    for column in categorical_columns:
        values = frame[column].fillna('__MISSING__').astype(str)
        counts = values.value_counts(dropna=False)
        frequency_maps[column] = (counts / len(frame)).to_dict()
        rare_values[column] = set(counts[counts < rare_min_count].index)
    return frequency_maps, rare_values

def apply_category_maps(frame, categorical_columns, frequency_maps, rare_values):
    df = frame.copy()
    for column in categorical_columns:
        values = df[column].fillna('__MISSING__').astype(str)
        df[f'{column}_Frequency'] = values.map(frequency_maps[column]).fillna(0).astype('float32')
        df[column] = values.where(~values.isin(rare_values[column]), '__RARE__')
    return df

train_base = add_base_features(train_raw.drop(columns=[TARGET]))
test_base = add_base_features(test_raw)
raw_categorical_columns = train_base.select_dtypes(include=['object', 'category']).columns.tolist()
frequency_maps, rare_values = fit_category_maps(train_base, raw_categorical_columns)
X_all = apply_category_maps(train_base, raw_categorical_columns, frequency_maps, rare_values)
X_test = apply_category_maps(test_base, raw_categorical_columns, frequency_maps, rare_values)
y = np.log1p(train_raw[TARGET].clip(lower=0))

assert all(feature in X_all.columns for feature in REQUIRED_FEATURES)
assert list(X_all.columns) == list(X_test.columns)
print(f'Features after engineering: {X_all.shape[1]}')

In [ ]:
def regression_metrics(y_log_true, y_log_pred):
    """Evaluate on original price scale; RMSLE equals RMSE on log1p prices."""
    y_true = np.expm1(y_log_true)
    y_pred = np.maximum(0, np.expm1(y_log_pred))
    return {
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'RMSLE': mean_squared_error(np.log1p(y_true), np.log1p(y_pred)) ** 0.5
    }

X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y, test_size=0.20, random_state=RANDOM_STATE
)
cat_columns = X_all.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_columns = [column for column in X_all.columns if column not in cat_columns]

# CatBoost requires categorical values to be strings and has native missing-value handling for numerics.
for frame in (X_train, X_valid, X_all, X_test):
    frame[cat_columns] = frame[cat_columns].fillna('__MISSING__').astype(str)

print(f'Numeric features: {len(numeric_columns)} | Categorical features: {len(cat_columns)}')

## Model comparison

The linear model uses only low-cardinality categories to provide an interpretable baseline. Random Forest uses ordinal encoding and is deliberately trained on a reproducible sample because it is substantially more expensive at this scale. CatBoost and LightGBM are the stronger gradient-boosting comparisons.

In [ ]:
results = []
if MODE == 'competition':
    low_cardinality = [c for c in cat_columns if X_train[c].nunique() <= 30]
    linear_columns = numeric_columns + low_cardinality
    linear_preprocessor = ColumnTransformer([('num', SimpleImputer(strategy='median'), numeric_columns), ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), low_cardinality)])
    linear_model = Pipeline([('preprocess', linear_preprocessor), ('model', LinearRegression())])
    linear_model.fit(X_train[linear_columns], y_train)
    results.append({'Model': 'Linear Regression', **regression_metrics(y_valid, linear_model.predict(X_valid[linear_columns]))})
    tree_preprocessor = ColumnTransformer([('num', SimpleImputer(strategy='median'), numeric_columns), ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_columns)])
    rf_model = Pipeline([('preprocess', tree_preprocessor), ('model', RandomForestRegressor(n_estimators=80, min_samples_leaf=2, max_features=0.7, n_jobs=-1, random_state=RANDOM_STATE))])
    rf_sample = X_train.sample(n=min(50_000, len(X_train)), random_state=RANDOM_STATE)
    rf_model.fit(rf_sample, y_train.loc[rf_sample.index])
    results.append({'Model': 'Random Forest (50k sample)', **regression_metrics(y_valid, rf_model.predict(X_valid))})
    lgb_train = X_train.copy()
    lgb_valid = X_valid.copy()
    for column in cat_columns:
        categories = pd.Index(sorted(X_all[column].astype(str).fillna('missing').unique()))
        lgb_train[column] = pd.Categorical(lgb_train[column].astype(str).fillna('missing'), categories=categories).codes
        lgb_valid[column] = pd.Categorical(lgb_valid[column].astype(str).fillna('missing'), categories=categories).codes
    lgb_model = LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63, subsample=0.9, colsample_bytree=0.8, random_state=RANDOM_STATE)
    lgb_model.fit(lgb_train, y_train, eval_set=[(lgb_valid, y_valid)], eval_metric='rmse', callbacks=[early_stopping(50), log_evaluation(100)])
    results.append({'Model': 'LightGBM', **regression_metrics(y_valid, lgb_model.predict(lgb_valid))})
else:
    print('Debug mode: skipping Linear Regression, Random Forest, and LightGBM.')
results_df = pd.DataFrame(results)
if not results_df.empty:
    display(results_df)

In [ ]:
# Debug mode deliberately trains exactly one CatBoost model.
if MODE == 'debug':
    catboost_model = CatBoostRegressor(loss_function='RMSE', iterations=700, learning_rate=0.05, depth=8, l2_leaf_reg=5, random_seed=RANDOM_STATE, verbose=100, allow_writing_files=False)
    catboost_model.fit(X_train, y_train, cat_features=cat_columns, eval_set=(X_valid, y_valid), early_stopping_rounds=80)
    catboost_pred = catboost_model.predict(X_valid)
    results.append({'Model': 'CatBoost (debug)', **regression_metrics(y_valid, catboost_pred)})
    results_df = pd.DataFrame(results)
    display(results_df)
else:
    print('Competition mode: CatBoost is trained in tuning, CV, and final fit only.')

## Five-fold validation and feature importance

Validation uses five shuffled folds with no target-derived features. Out-of-fold RMSLE is the selection metric. The settings below are intentionally modest for notebook practicality; increase `iterations` only after the baseline has completed.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_predictions = np.zeros(len(X_all))
fold_scores = []

for fold, (fit_idx, val_idx) in enumerate(cv.split(X_all) if MODE == 'competition' else [], start=1):
    model = CatBoostRegressor(
        loss_function='RMSE', iterations=650, learning_rate=0.05, depth=8, l2_leaf_reg=5,
        random_seed=RANDOM_STATE + fold, verbose=False, allow_writing_files=False
    )
    model.fit(X_all.iloc[fit_idx], y.iloc[fit_idx], cat_features=cat_columns,
              eval_set=(X_all.iloc[val_idx], y.iloc[val_idx]), early_stopping_rounds=80, verbose=200)
    oof_predictions[val_idx] = model.predict(X_all.iloc[val_idx])
    fold_rmsle = regression_metrics(y.iloc[val_idx], oof_predictions[val_idx])['RMSLE']
    fold_scores.append(fold_rmsle)
    print(f'Fold {fold}: RMSLE = {fold_rmsle:.5f}')
    del model
    gc.collect()

if MODE == 'competition':
    print(f'OOF RMSLE: {regression_metrics(y, oof_predictions)["RMSLE"]:.5f} +/- {np.std(fold_scores):.5f}')
else:
    print('Debug mode: cross-validation skipped.')

In [ ]:
if MODE == 'debug':
    importance = pd.DataFrame({
        'feature': X_all.columns,
        'importance': catboost_model.get_feature_importance()
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    plt.figure(figsize=(10, 8))
    sns.barplot(data=importance.head(25), x='importance', y='feature', color='teal')
    plt.title('CatBoost feature importance (debug holdout model)')
    plt.tight_layout()
    plt.show()
    display(importance.head(25))
else:
    importance = pd.DataFrame({'feature': X_all.columns, 'importance': np.nan})
    print('Competition mode: feature importance is generated after final training.')

## Feature selection and hyperparameter tuning

Debug mode uses importance-based feature trimming from the single holdout CatBoost. Competition mode keeps the full engineered feature set and runs a compact candidate search on a fixed sample without touching the test set.

In [ ]:
# Retain the smallest group of features that accounts for 98% of holdout importance.
importance['cumulative_importance'] = importance['importance'].cumsum() / importance['importance'].sum()
selected_features = importance.loc[importance['cumulative_importance'] <= 0.98, 'feature'].tolist()
if importance.iloc[0]['feature'] not in selected_features:
    selected_features = [importance.iloc[0]['feature']]
selected_cat_columns = [column for column in cat_columns if column in selected_features]
if MODE == 'competition':
    selected_features = X_all.columns.tolist()
    selected_cat_columns = cat_columns
print(f'Selected {len(selected_features)} of {X_all.shape[1]} features.')
display(importance[['feature', 'importance', 'cumulative_importance']])

tuning_candidates = [
    {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 3},
    {'depth': 8, 'learning_rate': 0.04, 'l2_leaf_reg': 5},
] if MODE == 'competition' else []
# A fixed 50k sample keeps competition-mode tuning practical while validation remains untouched.
tuning_fit = X_train.sample(n=min(TUNING_SAMPLE_SIZE, len(X_train)), random_state=RANDOM_STATE)
tuning_y = y_train.loc[tuning_fit.index]
tuning_results = []
for parameters in tuning_candidates:
    candidate = CatBoostRegressor(loss_function='RMSE', iterations=600, random_seed=RANDOM_STATE,
                                  verbose=False, allow_writing_files=False, **parameters)
    candidate.fit(tuning_fit[selected_features], tuning_y, cat_features=selected_cat_columns,
                  eval_set=(X_valid[selected_features], y_valid), early_stopping_rounds=80)
    candidate_pred = candidate.predict(X_valid[selected_features])
    tuning_results.append({**parameters, 'best_iteration': candidate.get_best_iteration() + 1,
                           'RMSLE': regression_metrics(y_valid, candidate_pred)['RMSLE']})
tuning_results = pd.DataFrame(tuning_results)
if tuning_results.empty:
    tuning_results = pd.DataFrame([{'depth': 8, 'learning_rate': 0.05, 'l2_leaf_reg': 5, 'best_iteration': 700, 'RMSLE': np.nan}])
else:
    tuning_results = tuning_results.sort_values('RMSLE').reset_index(drop=True)
display(tuning_results)
best_parameters = tuning_results.iloc[0][['depth', 'learning_rate', 'l2_leaf_reg']].to_dict()
best_parameters['depth'] = int(best_parameters['depth'])
print(f'Best tuning RMSLE: {tuning_results.iloc[0]["RMSLE"]:.5f}')

## Final model and Kaggle submission

The final CatBoost configuration uses the median best iteration from the validation model and the full training set. Predictions are transformed back to dollar scale, clipped at zero, and saved with `TransactionID` for Kaggle upload.

In [ ]:
best_iteration = max(300, int(tuning_results.iloc[0]['best_iteration']))
final_model = CatBoostRegressor(
    loss_function='RMSE', iterations=best_iteration, random_seed=RANDOM_STATE, verbose=200, allow_writing_files=False, **best_parameters
)
if MODE == 'competition':
    final_model.fit(X_all[selected_features], y, cat_features=selected_cat_columns)
    importance = pd.DataFrame({
        'feature': selected_features,
        'importance': final_model.get_feature_importance()
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    plt.figure(figsize=(10, 8))
    sns.barplot(data=importance.head(25), x='importance', y='feature', color='teal')
    plt.title('CatBoost feature importance (final competition model)')
    plt.tight_layout()
    plt.show()
    display(importance.head(25))
else:
    submission_path = OUTPUT_DIR / 'not_generated_in_debug_mode.csv'
    print('Debug mode: final retraining and submission generation are skipped.')

if MODE == 'competition':
    test_log_predictions = final_model.predict(X_test[selected_features])
    test_predictions = np.maximum(0, np.expm1(test_log_predictions))
    submission = pd.DataFrame({
        'TransactionID': test_raw['TransactionID'],
        TARGET: test_predictions
    })
    submission_path = OUTPUT_DIR / 'catboost_submission.csv'
    submission.to_csv(submission_path, index=False)

    assert len(submission) == len(test_raw)
    assert submission[TARGET].ge(0).all()
    print(f'Saved submission: {submission_path.resolve()}')
    display(submission.head())

## Project report

This concise report captures the data checks, model comparison, validation result, selected features, and submission location for the IITM project hand-in.

In [ ]:
if MODE == 'competition':
    oof_rmsle = regression_metrics(y, oof_predictions)['RMSLE']
    report_lines = [
    '# Heavy Equipment Selling Price Prediction — Run Report',
    '',
    f'- Training rows: {len(train_raw):,}; test rows: {len(test_raw):,}',
    f'- Input features after engineering: {X_all.shape[1]}',
    f'- Required engineered features present: {", ".join(REQUIRED_FEATURES)}',
    f'- OOF CatBoost RMSLE (5-fold): {oof_rmsle:.5f}',
    f'- Best tuned holdout RMSLE: {tuning_results.iloc[0]["RMSLE"]:.5f}',
    f'- Final selected features: {len(selected_features)}',
    f'- Submission file: {submission_path.resolve()}',
    '',
    '## Holdout model comparison',
    results_df.round(5).to_csv(index=False),
    '',
    '## Modelling decisions',
    '- Target is modelled as log1p(TargetValue), optimizing the RMSLE objective.',
    '- IQR outliers are reported but retained because extreme equipment can be valid transactions.',
    '- Category frequency and rare-category handling are fitted on training data only.',
    '- CatBoost is the final model because it natively handles missing and high-cardinality categorical values.'
    ]
    report_path = OUTPUT_DIR / 'model_report.md'
    report_path.write_text('\n'.join(report_lines), encoding='utf-8')
    print(f'Saved report: {report_path.resolve()}')
else:
    print('Debug mode: report generation skipped.')